In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: D:\dev\dermascan


In [2]:
from itertools import product

from pipeline.pipeline_orchestrator import PipelineOrchestrator
from pipeline.dataclasses.pipeline_config import PipelineConfig, BranchConfig, ModelConfig, TrainerConfig, TrainValTestSplitRatio, AugmentationConfig, CustomSplitConfig
from pipeline.dataclasses.enums import ModelType, FilterType, DatasetLoaderType, AugmentationType, LossType
from pipeline.dataclasses.training_configs import AdamWConfig, CosineAnnealingLRConfig

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold

test = StratifiedKFold

In [ ]:
CLASS_FREQS = (0.8501, 0.1494) # class 0: nv, class 1: melanoma
POS_WEIGHT = CLASS_FREQS[0] / CLASS_FREQS[1]
CE_CLASS_WEIGHTS = (1.0, POS_WEIGHT)

# Strict augmentations for RGB only. They keep the spatial semantics intact.
strict_augs = [
    AugmentationType.RANDOM_RESIZE_CROP,
]

base_models = [
    ModelType.RESNET_18,
    ModelType.EFFICIENTNET_V2_S,
    ModelType.INCEPTION_V3,
    ModelType.SQUEEZENET_1_1,
    ModelType.MOBILENET_V3_SMALL,
    ModelType.MOBILENET_V3_LARGE,
]
filter_type = FilterType.RGB
loader_type = DatasetLoaderType.RGB

experiment_specs = [
    {
        "experiment": "A",
        "output_nodes": 1,
        "loss_type": LossType.BCE_WITH_LOGITS,
        "loss": "BCEWithLogitsLoss",
        "pos_weight": 1.0,
        "class_weights": None,
        "purpose": "Baseline binary head",
    },
    {
        "experiment": "B",
        "output_nodes": 2,
        "loss_type": LossType.CROSS_ENTROPY,
        "loss": "CrossEntropyLoss",
        "pos_weight": 1.0,
        "class_weights": None,
        "purpose": "Equivalent binary head in softmax form",
    },
    {
        "experiment": "C",
        "output_nodes": 1,
        "loss_type": LossType.BCE_WITH_LOGITS,
        "loss": "BCEWithLogitsLoss",
        "pos_weight": POS_WEIGHT,
        "class_weights": None,
        "purpose": "Imbalance handling with pos_weight",
    },
    {
        "experiment": "D",
        "output_nodes": 2,
        "loss_type": LossType.CROSS_ENTROPY,
        "loss": "CrossEntropyLoss",
        "pos_weight": 1.0,
        "class_weights": CE_CLASS_WEIGHTS,
        "purpose": "Compare class weighting under softmax",
    },
]


In [ ]:
comparison_rows = []
for model, spec in product(base_models, experiment_specs):
    comparison_rows.append(
        {
            "experiment": spec["experiment"],
            "model": model.value,
            "output_nodes": spec["output_nodes"],
            "loss": spec["loss"],
            "purpose": spec["purpose"],
        }
    )

for row in comparison_rows:
    print(f"{row['experiment']} | {row['model']} | {row['output_nodes']} nodes | {row['loss']} | {row['purpose']}")

In [ ]:
def create_branch_config(model: ModelType, spec: dict) -> BranchConfig:
    branch_id = f"rgb_{model.value}_{spec['experiment'].lower()}_{spec['output_nodes']}node"
    return BranchConfig(
        branch_id=branch_id,
        sampler_weights=None,
        branch_model_config=ModelConfig(
            model_type=model,
            filter_type=filter_type,
            num_classes=spec["output_nodes"],
            pretrained=True,
            freeze_weights=False,
        ),
        trainer_config=TrainerConfig(
            batch_size=8,
            num_epochs=5,
            learning_rate=3e-4,
            loss_type=spec["loss_type"],
            pos_weight=spec["pos_weight"],
            class_weights=spec["class_weights"],
            optimizer=AdamWConfig(weight_decay=1e-4),
            scheduler=CosineAnnealingLRConfig(t_max=30, eta_min=1e-5),
        ),
        dataset_loader=loader_type,
        augmentation=AugmentationConfig(
            train=strict_augs,
            val=[AugmentationType.RESIZE],
        ),
    )

In [ ]:
branches = [create_branch_config(model, spec) for model, spec in product(base_models, experiment_specs)]

pipeline_config = PipelineConfig(
    pipeline_id="output_node_grid_resnet18_efficientnet_v2_s",
    data_path=proj_dir / "data",
    split_config=CustomSplitConfig(
        train_csv_path=Path(proj_dir / "data" / "train_df.csv"),
        val_csv_path=Path(proj_dir / "data" / "val_df.csv"),
        test_csv_path=Path(proj_dir / "data" / "test_df.csv"),
    ),
    seed=42,
    branches=branches,
)

print(f"Generated {len(branches)} branches for the experiment grid.")

In [ ]:
pipeline_runner = PipelineOrchestrator(
    pipeline_config=pipeline_config,
    project_root_path=proj_dir,
)

In [ ]:
run = pipeline_runner.run()